# Practice with PyPOTS, HuggingFace Datasets, and MAPIE

This notebook contains code for the experimentation with the pypots, datasets, and mapie Python libraries to help with the development of the project. It will consist of importing the data with datasets, constructing a DLinear model, and using MAPIE to create conformal prediction intervals. No forecasting.

In [ ]:
# Dataset imports
import datetime as dt
import numpy as np
import warnings
from datasets import load_dataset

warnings.simplefilter("ignore")

In [2]:
# import the dataset
dataset = load_dataset(
    "Dingdong-Inc/FreshRetailNet-50K",
    cache_dir="/home/gil/projects/uncertainty_aware_planner/data",
)

In [3]:
# converting the dataset to a Pandas dataframe
data = dataset['train'].to_pandas()

In [4]:
# sorting the data by primary keys Store ID, Product ID, and date (dt)
data = data.sort_values(by=PRIMARY_KEYS)
print(data.head())

     city_id  store_id  management_group_id  first_category_id  \
720        0         0                    2                 29   
721        0         0                    2                 29   
722        0         0                    2                 29   
723        0         0                    2                 29   
724        0         0                    2                 29   

     second_category_id  third_category_id  product_id          dt  \
720                  78                 82           4  2024-03-28   
721                  78                 82           4  2024-03-29   
722                  78                 82           4  2024-03-30   
723                  78                 82           4  2024-03-31   
724                  78                 82           4  2024-04-01   

     sale_amount                                         hours_sale  \
720          0.5  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, ...   
721          1.3  [0.0, 0.0, 0.0, 0.0, 0

In [5]:
# creating NumPy arrays from the list entries of the Pandas dataframe
hours_sale = np.array(data['hours_sale'].tolist())
hours_stock_status = np.array(data['hours_stock_status'].tolist())
print(hours_sale.shape)

(4500000, 24)


In [6]:

# reshaping the original sales data to be of the form [number_samples, sequence_length, number_features]
# which in these particular instances is reshaping scaled sales as well as
# stockout indicators into [unique-series x number_windows, days_per_window, number_hours]

# getting the number of unique store-product series in the dataset
series_num = data.shape[0] // HORIZON

# reshaping appropriately
hours_sale_origin = hours_sale.reshape(series_num*WINDOW_NUMBER, WINDOW_DURATION, 24)
hours_stock_status = hours_stock_status.reshape(series_num*WINDOW_NUMBER, WINDOW_DURATION, 24)

# only taking the hours that the store is open, as all others are not relevant for training
hours_sale_origin = hours_sale_origin[...,6:22]
hours_stock_status = hours_stock_status[...,6:22]
